# Evaluation Metrics for Poses and Trajectories

ATE, RPE, and the challenge-style **AA / mAA** are not three unrelated metrics.
Two of them are the *same residual* aggregated differently; the third is
deliberately different. This notebook is the bridge between the VO/SLAM
evaluation vocabulary and the Structure-from-Motion benchmark vocabulary.

| Metric | Tool | Meaning |
| ------- | --------- | ---------------------------------------- |
| **ATE** (APE) | `evo_ape` | Absolute Trajectory Error — global drift |
| **RPE** | `evo_rpe` | Relative Pose Error — local accuracy |
| **AA / mAA** | IMC scorer | Fraction of cameras inside a distance threshold, averaged over thresholds |

## 1. The one residual behind ATE and mAA

Both a monocular VO trajectory and an SfM reconstruction are only defined **up to
a similarity transform** — arbitrary origin, orientation, and scale. So neither
can be compared to ground truth directly. Both first solve

$$
T^\star = \arg\min_{s,R,\mathbf{t}} \sum_i \big\lVert\, C_{g,i} - (s R\,C_i + \mathbf{t}) \,\big\rVert^2 ,
\qquad s>0,\; R \in SO(3),
$$

then measure the **same per-camera residual**

$$
r_i \;=\; \big\lVert\, C_{g,i} - T^\star(C_i) \,\big\rVert .
$$

Everything after that is just a choice of how to summarise the $r_i$:

| | Summary of $\{r_i\}$ | Units | Hides |
|---|---|---|---|
| **ATE** | RMSE / median / max | metres | the shape of the distribution |
| **AA at threshold $\tau$** | $\dfrac{\#\{i : r_i < \tau\}}{N}$ | fraction | how badly the failures failed |
| **mAA** | mean of $A(\tau)$ over $\tau \in \mathcal{T}$ | fraction | same |

So, in one sentence:

> **mAA is the empirical CDF of the ATE residual, sampled at a few thresholds and averaged.**

ATE gives one number in metres and is dominated by the worst cameras. AA is
bounded in $[0,1]$, insensitive to how catastrophic an outlier is, and therefore
much better behaved as a leaderboard score — a single wildly misregistered camera
costs you $1/N$, not an unbounded RMSE.

**RPE is the odd one out.** It never applies a global alignment; it compares
relative motions over a window $\Delta$, which makes it gauge-invariant by
construction. There is no RPE analogue in SfM benchmarking, because an
unordered image collection has no temporal ordering to define $\Delta$.

In [1]:
import numpy as np
# Ten cameras, Sim(3)-aligned centre errors in metres -- ONE catastrophic outlier.
err = np.array([0.12, 0.05, 0.31, 0.44, 0.09, 0.62, 1.4, 0.27, 0.18, 6.0])

print(f"ATE RMSE = {np.sqrt(np.mean(err**2)):.2f} m    <- one number, outlier-dominated")
thresholds = [0.5, 1, 2, 4]
for tau in thresholds:
    print(f"AA @ {tau:>3} m = {np.mean(err < tau):.0%}")
maa = np.mean([np.mean(err < tau) for tau in thresholds])
print(f"mAA       = {maa:.0%}   <- same residuals: 9 good cameras still count")

ATE RMSE = 1.97 m    <- one number, outlier-dominated
AA @ 0.5 m = 70%
AA @   1 m = 80%
AA @   2 m = 90%
AA @   4 m = 90%
mAA       = 82%   <- same residuals: 9 good cameras still count


## 2. Same Umeyama, different robustness

The closed form for $T^\star$ — centroids, cross-covariance, SVD, scale,
translation — is identical in both worlds (Umeyama 1991 / Horn 1987). It is
derived in
[Trajectory analysis §4.1](visual_odometry/trajectory_analysis.ipynb#41-sim3--umeyama-1991-for-monocular).

What differs is **how it is fitted**:

| | Input | Fit |
|---|---|---|
| `evo` / `rpg_trajectory_evaluation` | ordered, timestamped trajectory | plain least squares over all poses |
| IMC scorer | unordered bag of camera centres | **RANSAC** over triplets, keep the model with most inliers |

This is not cosmetic. A VO trajectory drifts *smoothly*, so every pose is roughly
consistent and least squares is fine. An SfM reconstruction fails
*discontinuously* — mirrored sub-clusters, cameras dumped at the origin,
whole components misregistered — and a non-robust fit gets dragged off by a
handful of them, destroying the score for cameras that were actually correct.

If you ever evaluate an SfM result with `evo`-style least-squares alignment, you
will under-report accuracy for exactly this reason.

In [2]:
# 12 cameras on a line; the reconstruction is PERFECT except two cameras
# collapsed to the origin. Align with least squares vs. RANSAC
# (translation-only here, to keep the effect visible in one number).
gt_c  = np.column_stack([np.arange(12.0), np.zeros(12), np.zeros(12)])
est_c = gt_c.copy(); est_c[[5, 9]] = 0.0        # the two failures

def maa_of(t):
    e = np.linalg.norm(est_c + t - gt_c, axis=1)
    return np.mean([np.mean(e < tau) for tau in (0.5, 1, 2, 4)])

t_ls = gt_c.mean(0) - est_c.mean(0)             # least squares: uses ALL cameras
best = max(range(12), key=lambda i: np.sum(     # RANSAC: best 1-camera hypothesis
    np.linalg.norm(est_c + (gt_c[i] - est_c[i]) - gt_c, axis=1) < 0.5))
t_ransac = gt_c[best] - est_c[best]

print(f"least squares: mAA = {maa_of(t_ls):.0%}   (2 outliers dragged the fit off)")
print(f"RANSAC       : mAA = {maa_of(t_ransac):.0%}   (outliers ignored; 10 exact cameras)")

least squares: mAA = 44%   (2 outliers dragged the fit off)
RANSAC       : mAA = 83%   (outliers ignored; 10 exact cameras)


## 3. The convention trap: $C = -R^\top t$

Every one of these metrics is defined on **camera centres in the world frame**,
not on the translation vector you store.

$$
X_{\text{cam}} = R\,X_{\text{world}} + t
\qquad\Longrightarrow\qquad
C = -R^{\top} t .
$$

A submission or a trajectory file that puts $t$ where $C$ belongs is not
"slightly off" — it is wrong by the full camera position, and scores ~0 while
looking perfectly well-formed. The world→cam vs cam→world bookkeeping is worked
through in [Understanding poses](understanding_poses.md).

In [3]:
# A single perfect world->cam pose: camera 3 m east of the origin, yawed 90 deg.
R = np.array([[0., 1, 0], [-1, 0, 0], [0, 0, 1]])   # world->cam rotation
C_true = np.array([3.0, 0, 0])                      # true camera centre
t = -R @ C_true                                     # what the pose file stores

C_wrong = t                                         # the classic mistake
C_right = -R.T @ t
print(f"stored t         = {t}")
print(f"t used as centre = {C_wrong}  -> {np.linalg.norm(C_wrong - C_true):.1f} m 'error' on a perfect pose")
print(f"-R^T t           = {C_right}  -> {np.linalg.norm(C_right - C_true):.1f} m")

stored t         = [0. 3. 0.]
t used as centre = [0. 3. 0.]  -> 4.2 m 'error' on a perfect pose
-R^T t           = [3. 0. 0.]  -> 0.0 m


## 4. Two different metrics are both called "mAA"

This is the single most common source of confusion when reading matching papers,
because the Image Matching Challenge changed families mid-life and kept the name.

| | **Relative-pose mAA** (IMC ~2019–2023) | **Camera-centre mAA** (IMC 2024, 2025) |
|---|---|---|
| Unit of evaluation | image **pair** | individual **camera** |
| Error measured | rotation geodesic on $SO(3)$, in degrees, **and** translation-*direction* angle in $[0,180]^\circ$ | Euclidean distance between 3D centres, in metres |
| Thresholds | angular, e.g. $10^\circ$ | metric, e.g. $\{0.5,1,2,4\}$ m |
| Global alignment | none | Sim(3), RANSAC-fitted |
| Scale | invisible (unit vectors) | explicitly recovered as $s$ |
| Combination rule | $\max(\text{err}_{\text{rot}}, \text{err}_{\text{trans}}) < \tau$ | single distance threshold |

Both are "mean Average Accuracy" — cumulative accuracy at increasing thresholds,
averaged. Only the error function changes.

Note that the rotation error of the *first* family,

$$
\theta = \arccos\!\Big(\frac{\operatorname{tr}(R_a^{\top} R_b) - 1}{2}\Big),
$$

is the **same geodesic distance** used as a *training* loss in
[VO loss functions §1.1](vo_loss_functions.ipynb) — measured in degrees for
evaluation, minimised in radians for training.

**Where each is written up:**

- Relative-pose mAA, and its relationship to mAP —
  [PythonTutorial · metrics_and_scoring_methods.ipynb §1.7.1](https://github.com/behnamasadi/PythonTutorial/blob/master/machine_learning/metrics_and_scoring_for_model_evaluation/metrics_and_scoring_methods.ipynb#171-maa-mean-average-accuracy)
- Camera-centre mAA, with clustering and a fully worked Umeyama alignment —
  `PyTorchTutorial/kaggle_structure/competitions/image_matching_2025/imc2025_scoring_explained.ipynb`

## 5. What is *not* related

The training-side notebooks in this repo — [photometric reprojection
loss](photometric_reprojection_loss.ipynb), [SSIM](ssim.ipynb), [edge-aware depth
smoothness](edge_aware_depth_smoothness.ipynb) — are dense, differentiable,
**image-space** objectives used to train a depth+pose network.

None of them appear in any of the metrics above. A benchmark scorer receives
poses, never pixels. The only connection is a pipeline one: those losses train a
network whose poses you could then evaluate with ATE or submit to a challenge.
(In practice the challenge leaderboards are dominated by classical SfM —
matching plus COLMAP/GLOMAP — not by learned VO.)

The pose-side losses are the exception: the $SO(3)$ geodesic and SE(3) losses in
[VO loss functions](vo_loss_functions.ipynb) are the same functions as the
evaluation errors above, just used as gradients instead of as reports.

## 6. See also

- [Trajectory analysis](visual_odometry/trajectory_analysis.ipynb) — ATE/RPE in
  depth, Sim(3)/SE(3)/yaw-only/posyaw alignment, sub-trajectory drift, and the
  `evo` / `rpg_trajectory_evaluation` tooling
- [Understanding poses](understanding_poses.md) — KITTI pose format, $T_{w,i}$,
  and the world→cam conversion
- [VO loss functions](vo_loss_functions.ipynb) — the training-time counterparts